In [1]:
from freelance_finance_dl.dataloader import FinanceTransactionDataset, get_data_loader
from freelance_finance_dl.model import TransactionAutoencoder

import pandas as pd
import torch
import torch.nn as nn

In [2]:
raw_data_path = "../data/budgetwise_finance_dataset.csv"

df = pd.read_csv(raw_data_path)

print("Raw dataset shape:", df.shape)

df.head()

Raw dataset shape: (15900, 9)


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaN,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad,NaN


In [3]:
print("Transaction category counts:")

df["category"].value_counts()

Transaction category counts:


category
FOOD             583
Food             579
RENT             574
Rent             571
Foods            567
food             567
Others           554
Fod              535
rent             534
Foodd            525
Rentt            517
Rnt              516
Freelance        490
Investment       482
Bonus            442
Salary           430
Travel           422
Traval           411
TRAVEL           403
travel           381
Travl            368
Entertain        349
Entrtnmnt        340
Utility          330
Utilties         329
entertainment    327
Entertainment    319
Utilities        317
utilities        313
Utlities         300
education        284
Education        259
Educaton         258
EDU              235
health           159
Helth            157
Health           156
HEALTH           151
Savings          110
SAVINGS          101
savings           99
Saving            85
Other             50
others            48
Misc              45
OTHERS            43
Name: count, dtype: int64

In [4]:
sequence_length = 5

dataset = FinanceTransactionDataset(
    csv_file=raw_data_path,
    sequence_length=sequence_length,
)

print("Number of category-specific sequences:", len(dataset))

Number of category-specific sequences: 1104


In [5]:
x, y = dataset[0]

print("Input sequence:")
print(x)

print("\nTarget sequence:")
print(y)

print("\nInput shape:", x.shape)
print("Target shape:", y.shape)

print("\nMetadata:")
print(dataset.get_sample_metadata(0))

Input sequence:
tensor([[1.0000],
        [1.0000],
        [0.3603],
        [0.8227],
        [0.0000]])

Target sequence:
tensor([[1.0000],
        [1.0000],
        [0.3603],
        [0.8227],
        [0.0000]])

Input shape: torch.Size([5, 1])
Target shape: torch.Size([5, 1])

Metadata:
{'user_id': 'U001', 'category': 'entertainment', 'sequence': array([9484., 9484., 5385., 8348., 3076.])}


In [6]:
loader = get_data_loader(
    csv_file=raw_data_path,
    batch_size=32,
    sequence_length=5,
    shuffle=True,
)

batch_x, batch_y = next(iter(loader))

print("Batch input shape:", batch_x.shape)
print("Batch target shape:", batch_y.shape)

Batch input shape: torch.Size([32, 5, 1])
Batch target shape: torch.Size([32, 5, 1])


In [7]:
model = TransactionAutoencoder(sequence_length=5, hidden_dim=64, latent_dim=16)

reconstructed = model(batch_x)

print("Model architecture:")
print(model)
print("\nOriginal batch shape:", batch_x.shape)
print("Reconstructed batch shape:", reconstructed.shape)

Model architecture:
TransactionAutoencoder(
  (encoder_lstm): LSTM(1, 64, batch_first=True)
  (encoder_fc): Linear(in_features=64, out_features=16, bias=True)
  (decoder_fc): Linear(in_features=16, out_features=64, bias=True)
  (decoder_lstm): LSTM(64, 64, batch_first=True)
  (output_fc): Linear(in_features=64, out_features=1, bias=True)
)

Original batch shape: torch.Size([32, 5, 1])
Reconstructed batch shape: torch.Size([32, 5, 1])


In [8]:
loss_fn = nn.MSELoss()

loss = loss_fn(reconstructed, batch_y)

print("Example reconstruction loss:", loss.item())

Example reconstruction loss: 0.32752344012260437


In [9]:
reconstruction_errors = ((batch_x - reconstructed) ** 2).mean(dim=(1, 2))

print("First 10 reconstruction errors:")
print(reconstruction_errors[:10])

threshold = reconstruction_errors.mean() + 2 * reconstruction_errors.std()

print("Example anomaly threshold:", threshold.item())

anomaly_flags = reconstruction_errors > threshold

print("First 10 anomaly flags:")
print(anomaly_flags[:10])

First 10 reconstruction errors:
tensor([0.4839, 0.4140, 0.2328, 0.3371, 0.2588, 0.2037, 0.5683, 0.4275, 0.2465,
        0.3095], grad_fn=<SliceBackward0>)
Example anomaly threshold: 0.5563039779663086
First 10 anomaly flags:
tensor([False, False, False, False, False, False,  True, False, False, False])
